# Intro by Thant
### Data Structure Overview (NASA C-MAPSS FD001)
---
The FD001 dataset consists of space-delimited text files. Each row represents a "snapshot" of an engine's operational state during a specific time cycle.


*   Column 1: Unit Number (ID)

    Identifies the specific engine. FD001 contains data for 100 unique units (ID 1 to 100).

*   Column 2: Time in Cycles

    Represents the discrete operational time steps. One "cycle" typically covers a full flight profile (Start → Take-off → Cruise → Landing). In the training set, the engine runs until it reaches a fault state. The maximum cycle for a unit is its total lifespan.

*   Columns 3 - 5: Operational Settings

    Environmental and setting parameters: Altitude, Mach Number, and Sea Level Pressure. In FD001, these are mostly constant as the engines operate under a single condition.

*   Columns 6 - 26: Sensor Measurements (1 to 21)

    Raw readings from 21 different sensors. As an engine degrades, some sensors will show an increasing trend, while others will decrease.

[Important] The Missing Target (RUL)

There is no "RUL" (Remaining Useful Life) column in the raw files.

*   For Training: We must calculate the label manually: RUL=Max_Cycle unit−Current_Cycle.

*   For Testing: The data is "truncated" before failure. We must predict how many cycles remain before the engine fails, based on the sensor patterns we learned during training.





In [ ]:
import os

In [ ]:
!ls "/content/drive/MyDrive/UWA-Capstone-Group17"

train_FD001.txt


In [ ]:
# @title 1. Data Downloading

from google.colab import drive
import os
drive.mount('/content/drive')

path = "/content/drive/MyDrive/UWA-Capstone-Group17"

if os.path.exists(path):
    os.chdir(path)
    print(f"Switch to {os.getcwd()}")

    !ls -lh train_FD001.txt
else:
    print(f"Failed to find {path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Switch to /content/drive/MyDrive/UWA-Capstone-Group17
-rw------- 1 root root 3.4M Jul 25  2019 train_FD001.txt


In [ ]:
# @title 2. Utils
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

#  Function to Load and Clean Data (Add RUL and Drop Features)
def preprocess_raw_data(file_path):
    """
    Load NASA Turbofan dataset, calculate RUL, and drop constant features.
    """
    cols = ['unit','cycle','op1','op2','op3'] + [f'sensor{i}' for i in range(1,22)]
    df = pd.read_csv(file_path, sep='\s+', header=None, names=cols)

    # Calculate Remaining Useful Life (RUL)
    # RUL = (max cycles for that unit) - (current cycle)
    max_cycle = df.groupby('unit')['cycle'].transform('max')
    df['RUL'] = max_cycle - df['cycle']

    # Drop constant/low-variance features for FD001
    drop_cols = ['op3','sensor1','sensor5','sensor10','sensor16','sensor18','sensor19']
    df_clean = df.drop(columns=drop_cols)

    return df_clean

#  Function to Split Data by Unit ID
def split_data_by_unit(df, test_size=0.2):
    """
    Split data ensuring that all cycles of a single engine stay together.
    This prevents temporal data leakage.
    """
    units = df['unit'].unique()
    train_units, val_units = train_test_split(units, test_size=test_size, random_state=42)

    train_df = df[df['unit'].isin(train_units)].copy()
    val_df = df[df['unit'].isin(val_units)].copy()

    return train_df, val_df

#  Function to Scale Features
def scale_features(train_df, val_df):
    """
    Scale features using StandardScaler.
    Fit ONLY on training data to avoid data leakage.
    """
    # Identify feature columns (exclude unit, cycle, and target RUL)
    feature_cols = [c for c in train_df.columns if c not in ['unit', 'cycle', 'RUL']]

    scaler = StandardScaler()

    # Fit and transform training data
    train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])

    # Transform validation data using training mean/std
    val_df[feature_cols] = scaler.transform(val_df[feature_cols])

    return train_df, val_df, scaler

<>:13: SyntaxWarning: invalid escape sequence '\s'
<>:13: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_160/3265372738.py:13: SyntaxWarning: invalid escape sequence '\s'
  df = pd.read_csv(file_path, sep='\s+', header=None, names=cols)


In [ ]:
# @title 3. Data cleaning and splitting

# Step 1: Clean
df_processed = preprocess_raw_data("train_FD001.txt")

# Step 2: Split (80% Train, 20% Validation)
train_raw, val_raw = split_data_by_unit(df_processed, test_size=0.2)

# Step 3: Scale
train_final, val_final, fitted_scaler = scale_features(train_raw, val_raw)

# Step 4: Save to CSV
train_final.to_csv("train_split_scaled.csv", index=False)
val_final.to_csv("val_split_scaled.csv", index=False)

print(f"Successfully saved processed files!")
print(f"Train shape: {train_final.shape}, Val shape: {val_final.shape}")

Successfully saved processed files!
Train shape: (16561, 20), Val shape: (4070, 20)


In [ ]:
# @title 4. Dataset Inspection

# 1. Display the first few rows of the final processed training set
print(" Processed Training Set (Scaled & RUL added) ")
display(train_final.head(10))

# 2. Check the statistics to confirm scaling (Mean should be near 0, Std near 1)
print("\n Feature Statistics (Verification) ")

# We only check the sensor columns, not unit, cycle or RUL
check_cols = [c for c in train_final.columns if c not in ['unit', 'cycle', 'RUL']]
display(train_final[check_cols].describe().loc[['mean', 'std', 'min', 'max']])


 Processed Training Set (Scaled & RUL added) 


,unit,cycle,op1,op2,sensor2,sensor3,sensor4,sensor6,sensor7,sensor8,sensor9,sensor11,sensor12,sensor13,sensor14,sensor15,sensor17,sensor20,sensor21,RUL
192,2,1,-0.818512,2.043793,-1.581191,-1.092662,-1.963014,-7.193194,1.318090,-1.221017,-0.471297,-2.294365,1.242127,-0.500147,-0.315944,-1.373754,-1.426246,0.678684,1.564436,286
193,2,2,1.966124,-1.031739,-1.721722,-0.565155,-1.756664,0.139020,1.591518,-1.645675,-0.621789,-1.127747,1.745551,-1.615806,-0.654442,-0.675217,-0.779529,1.345844,1.098895,285
194,2,3,0.824880,1.018616,-2.263772,-0.356453,-1.106382,-7.193194,2.013053,-0.796358,-0.492481,-1.203012,1.582278,-1.476349,-0.169925,-1.648369,-1.426246,1.623827,1.252524,284
195,2,4,1.600926,-1.373465,-2.002785,-1.041719,-1.427619,0.139020,0.999091,-1.645675,-0.326102,-1.654606,1.459824,-2.313093,-0.177072,-1.072476,-1.426246,1.735021,1.975974,283
196,2,5,0.231433,1.360341,-1.902406,-1.883102,-0.709298,-7.193194,1.990267,-0.937911,-0.319923,-1.090114,1.160491,-2.173636,-0.369552,-1.475068,-2.072963,2.013004,1.237627,282
197,2,6,-0.453314,1.360341,-2.765670,-2.134530,-1.351772,0.139020,1.830767,-0.796358,-0.499983,-1.842771,1.881611,-1.476349,-0.523740,-1.653702,-0.779529,1.846214,1.271145,281
198,2,7,0.048833,-0.690013,-1.300128,-0.492849,-0.917879,0.139020,1.671268,-0.654805,-0.561769,-1.654606,0.983613,-0.779062,-0.386911,-1.277771,-1.426246,1.568231,1.688270,280
199,2,8,0.687930,-1.373465,-0.256180,-0.012998,-1.504582,0.139020,1.454804,-1.221017,-0.422752,-1.278278,1.840793,-1.894721,-0.305733,-1.491065,-1.426246,1.623827,1.374495,279
200,2,9,0.779230,-1.373465,-1.400508,-1.396678,-1.546968,-7.193194,1.500375,-1.079464,-0.605901,-1.504075,1.337370,-1.615806,0.070547,-1.472402,-2.072963,1.345844,1.834450,278
201,2,10,-2.051056,0.676890,-1.380432,-0.676901,-1.563699,-7.193194,1.432018,-1.504122,-0.287707,-1.240645,0.779522,-1.755264,-0.268972,-1.168458,-1.426246,2.124197,2.886572,277



 Feature Statistics (Verification) 


,op1,op2,sensor2,sensor3,sensor4,sensor6,sensor7,sensor8,sensor9,sensor11,sensor12,sensor13,sensor14,sensor15,sensor17,sensor20,sensor21
mean,4.290458e-19,-2.231038e-17,-2.197573e-14,2.314788e-14,-4.397719e-15,-6.504226e-12,-4.517681e-14,-1.117486e-12,-6.902907e-14,-3.026918e-14,1.055058e-13,9.343313e-13,3.926198e-15,5.130101e-15,-1.643846e-14,7.115424e-14,2.569813e-14
std,1.000030e+00,1.000030e+00,1.000030e+00,1.000030e+00,1.000030e+00,1.000030e+00,1.000030e+00,1.000030e+00,1.000030e+00,1.000030e+00,1.000030e+00,1.000030e+00,1.000030e+00,1.000030e+00,1.000030e+00,1.000030e+00,1.000030e+00
min,-3.968346e+00,-2.056917e+00,-2.946353e+00,-3.196118e+00,-2.642296e+00,-7.193194e+00,-4.013752e+00,-2.778098e+00,-1.927223e+00,-2.595427e+00,-3.710470e+00,-3.010380e+00,-2.244824e+00,-3.042777e+00,-3.366398e+00,-3.769046e+00,-3.689657e+00
max,3.974714e+00,2.043793e+00,3.718851e+00,4.341812e+00,3.637440e+00,1.390203e-01,3.061192e+00,6.564392e+00,7.908111e+00,3.689256e+00,2.670761e+00,6.472724e+00,7.648730e+00,3.806617e+00,4.394210e+00,3.402920e+00,3.053235e+00


## Tip : How to Use the Processed Dataset?

After runing this section, everyone will get
*   Training Set: `train_split_scaled.csv` (80% of units)
*   Validation Set: `val_split_scaled.csv` (20% of units)

The datasets are already cleaned, split by unit, and scaled based on the training set distribution.

Usage example:


```
# Load the processed datasets
train_df = pd.read_csv("train_split_scaled.csv")
val_df = pd.read_csv("val_split_scaled.csv")

# Define feature and target columns
target_col = 'RUL'
feature_cols = [c for c in train_df.columns if c not in ['unit', 'cycle', 'RUL']]

# Prepare X and y
X_train, y_train = train_df[feature_cols], train_df[target_col]
X_val, y_val = val_df[feature_cols], val_df[target_col]

# Check if needed
print(f"Features being used ({len(feature_cols)}): {feature_cols}")
```






# 1. k-NN by Thant

# 2. DevNet by Swapnil

#AutoEncoder by Zeyang

In this part I implemented an AutoEncoder model to detect anomalies in the sensor data. It consists of two main components, an encoder and a decoder. The encoder compresses the input features into a lower-dimensional latent representation, and the decoder reconstructs the original data from this compressed data. This model is trained to minimize the reconstruction error between the input and output.

In [ ]:
import numpy as np

# Sensor features
feature_cols = [c for c in train_final.columns if c not in ['unit','cycle','RUL','op1','op2']]

X = train_final[feature_cols].values
print(X.shape)

import tensorflow as tf
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model

input_dim = X.shape[1]

input_layer = Input(shape=(input_dim,))

# Encoder
encoded = Dense(64, activation='relu')(input_layer)
encoded = Dense(32, activation='relu')(encoded)
latent = Dense(16, activation='relu')(encoded)

# Decoder
decoded = Dense(32, activation='relu')(latent)
decoded = Dense(64, activation='relu')(decoded)
output = Dense(input_dim, activation='linear')(decoded)

autoencoder = Model(input_layer, output)

autoencoder.compile(
    optimizer='adam',
    loss='mse'
)

autoencoder.summary()

history = autoencoder.fit(
    X,
    X,
    epochs=50,
    batch_size=256,
    validation_split=0.2,
    verbose=1
)

reconstruction = autoencoder.predict(X)

error = np.mean(np.square(X - reconstruction), axis=1)

train_final['anomaly_score'] = error

train_final[['cycle','RUL','anomaly_score']].head()


threshold = np.mean(error) + 3*np.std(error)

train_final['anomaly'] = train_final['anomaly_score'] > threshold

#PreNet by Zeyang

PreNet is a feature preprocessing model before the main model, it is a small neural network which transform or refine the input features. It is not a standalone algorithm. So I combined it with the Autoencoder to evaluate whether it can help the model learn more information and improve the detection performance.

In [ ]:
from tensorflow.keras.layers import Dropout

input_layer = Input(shape=(input_dim,))

# PreNet
x = Dense(128, activation='relu')(input_layer)
x = Dropout(0.3)(x)
x = Dense(64, activation='relu')(x)

# Encoder
x = Dense(32, activation='relu')(x)
latent = Dense(16, activation='relu')(x)

# Decoder
x = Dense(32, activation='relu')(latent)
x = Dense(64, activation='relu')(x)
output = Dense(input_dim, activation='linear')(x)

model_prenet = Model(input_layer, output)

model_prenet.compile(
    optimizer='adam',
    loss='mse'
)

model_prenet.summary()

model_prenet.fit(
    X,
    X,
    epochs=50,
    batch_size=256,
    validation_split=0.2
)

reconstruction2 = model_prenet.predict(X)

error2 = np.mean((X - reconstruction2)**2, axis=1)

print("Autoencoder mean error:", error.mean())
print("PreNet+AE mean error:", error2.mean())

# IF/EIF by swapnil

# LOF/CBLOF by Shreyan

#XGBOD by Shreyan

#Anomaly Transformer by Julie

#SVM/OCSVM by Julie